# 0. Install and import the required dependencies

**You may add or remove based on your assigned model!**

In [1]:
!pip install -U tokenizers transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata

# 1. Authentication & Model selection

**Retrieve the Hugging Face token securely from Colab's "Secrets" tab (the key icon on the left).**

In [3]:
try:
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("WARNING: 'HF_TOKEN' not found in Colab Secrets.")
    hf_token = None

**CHANGE THIS TO YOUR ASSIGNED MODEL**

In [4]:
# A smaller model that fits on a free Colab GPU
model_name = "Qwen/Qwen2.5-7B-Instruct"

# 2. Hardware optimization (Quantization)

In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # What is loaded in 4 bit? why?
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 3. Load the tokenizer & model

In [6]:
print(f"Loading Tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token,
    # trust_remote_code=True # Does your model need it?
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" # What about right?

Loading Tokenizer for Qwen/Qwen2.5-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
print("Loading Model on Colab T4 GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    #  trust_remote_code=True, # Does your model need it?
    quantization_config=bnb_config, # from section 2 above
    torch_dtype=torch.float16
)

Loading Model on Colab T4 GPU...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

# 4. INFERENCE & HYPERPARAMETER TUNING

**Design the prompt (Does this design/technique have a name?)**

In [20]:
prompt = [
    {
        "role": "system",
        "content": "You are a helpful software assistant. Your job is to explain the functionality of the provided code in simple terms. Throughout the code, for any loops "+
        " iterations go trace the variables in each iteration before providing the final answer"
    },
    {
        "role": "user",
        "content": """Please analyze the following source code:

        <source_code>
        public class Task {

            public static void main(String[] args) {
                String id1   = "AMQ-2104";
                double files = 8.0;
                String id2   = "AMQ-317";
                double dmm   = 0.45;

                int n = Integer.parseInt(id1.split("-")[1])
                      + Integer.parseInt(id2.split("-")[1]);

                int digits = 0;
                while (n > 0) {
                    n = n / 10;
                    digits++;
                }

                int impact = (int) (files * dmm);

                System.out.println("Combined digits: " + digits);
                System.out.println("Impact: " + impact);
            }
        }
        </source_code>"""
    }
]

**Convert the prompt into tokens**

In [21]:
inputs = tokenizer.apply_chat_template(
    prompt,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

**Understand these hyperparameters and then tune them for better inference**

In [22]:
print("Generating response...\n")
outputs = model.generate(
    **inputs,
    max_new_tokens=1024, # which tokens?
    temperature=0.3, # ?
    top_p=0.8, # ?
    do_sample=True, # What if this toggle is false?
    pad_token_id=tokenizer.eos_token_id
)

Generating response...



**Convert the output back from tokens into human-readable text and display it**

In [23]:
input_length = inputs['input_ids'].shape[1]
response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

print("Model output:")
print(response)

Model output:
Sure! Let's break down the provided Java code step by step, including tracing the variables through each iteration of the loop.

### Code Breakdown

1. **Initialization of Variables:**
    ```java
    String id1   = "AMQ-2104";
    double files = 8.0;
    String id2   = "AMQ-317";
    double dmm   = 0.45;
    ```
    - `id1` and `id2` are strings containing IDs.
    - `files` is a double representing the number of files.
    - `dmm` is a double representing a value related to DMM (likely a damage or maintenance measure).

2. **Combining IDs:**
    ```java
    int n = Integer.parseInt(id1.split("-")[1])
          + Integer.parseInt(id2.split("-")[1]);
    ```
    - `id1.split("-")` splits the string `"AMQ-2104"` into an array `["AMQ", "2104"]`.
    - `Integer.parseInt(id1.split("-")[1])` converts the second element (`"2104"`) to an integer, resulting in `2104`.
    - Similarly, `id2.split("-")` splits the string `"AMQ-317"` into `["AMQ", "317"]`, and `Integer.parseInt(id2.